# Experiment Template

**Only edit Cell 1 (CONFIG).** Everything else is boilerplate handled by `src/pipeline.py`.

Workflow:
1. Copy this notebook, rename it (e.g. `18_my_experiment.ipynb`)
2. Change `experiment_name` and adjust `encode_pairs`, `drop_cols`, and model params in CONFIG
3. Run all cells
4. View all experiment runs: `mlflow ui` in the project root → open http://localhost:5000

In [1]:
# ── EDIT ONLY THIS CELL ────────────────────────────────────────────────────
CONFIG = {
    "experiment_name": "v16-pipeline-test",  # Reproduces v16 — expected OOF ~$21,552

    "data": {
        "train_path": "../../data/raw/train.csv",
        "test_path":  "../../data/raw/test.csv",
    },

    "features": {
        "encode_pairs": [
            ("town",           "town_median_price",           1),
            ("postal_sector",  "postal_sector_median_price",  1),
            ("flat_model",     "flat_model_median_price",     1),
            ("town_flat_type", "town_flat_type_median_price", 3),
            ("Tranc_Year",     "year_median_price",           1),
        ],
        "drop_cols": [],
    },

    "models": {
        "active": ["xgb", "lgbm", "et", "catboost"],

        "xgb": {
            "n_estimators": 2000, "max_depth": 7, "learning_rate": 0.05,
            "subsample": 0.9, "colsample_bytree": 0.4, "min_child_weight": 7,
            "reg_alpha": 0.01, "reg_lambda": 1.5,
            "random_state": 42, "n_jobs": -1, "verbosity": 0, "tree_method": "hist",
        },
        "lgbm": {
            "n_estimators": 1000, "max_depth": 12, "num_leaves": 300,
            "learning_rate": 0.03, "subsample": 0.8, "colsample_bytree": 0.5,
            "min_child_samples": 20, "reg_alpha": 0, "reg_lambda": 0.5,
            "random_state": 42, "n_jobs": -1, "verbosity": -1,
        },
        "et": {
            "n_estimators": 300, "min_samples_split": 10, "min_samples_leaf": 1,
            "max_features": 0.8, "max_depth": 20,
            "random_state": 42, "n_jobs": -1,
        },
        "catboost": {
            "iterations": 1000, "depth": 7, "learning_rate": 0.08,
            "l2_leaf_reg": 1, "colsample_bylevel": 0.7, "min_child_samples": 5,
            "loss_function": "RMSE", "random_seed": 42, "verbose": 0,
        },
        "ridge_alphas": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    },

    "cv": {"n_splits": 5, "random_state": 42},

    "output": {
        "submission_path": "../../outputs/submissions/sub_v16_pipeline_test.csv",
        "sample_path":     "../../outputs/submissions/sample_sub_reg.csv",
    },
}

In [2]:
import sys
sys.path.insert(0, "../../")

from src.pipeline import run_pipeline

results = run_pipeline(CONFIG)

/Users/tehmenghai/miniconda3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026/04/29 10:33:57 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/04/29 10:33:57 INFO mlflow.store.db.utils: Updating database tables


2026/04/29 10:33:57 INFO mlflow.tracking.fluent: Experiment with name 'v16-pipeline-test' does not exist. Creating a new experiment.



Experiment: v16-pipeline-test


...

    2  $       22,278  $       22,248  $       23,697  $       23,062


    3  $       21,643  $       21,622  $       23,268  $       22,600


    4  $       21,899  $       21,853  $       23,499  $       22,565


    5  $       21,777  $       21,679  $       23,354  $       22,556
-----------------------------------------------------------------------
 Mean  $       21,821  $       21,776  $       23,372  $       22,607

Ridge alpha sweep:
     alpha    OOF RMSE  xgb-coef      lgbm-coef     et-coef       catboost-coef
-------------------------------------------------------------------------------
     0.001  $    21,552        0.3431        0.3673        0.1198        0.1725
     0.010  $    21,552        0.3431        0.3672        0.1198        0.1726
     0.100  $    21,552        0.3430        0.3666        0.1201        0.1729
     1.000  $    21,553        0.3418        0.3612        0.1233        0.1765
    10.000  $    21,558        0.3268        0.3282        0.1470        0.2008
   100.000  $    21,594        0.2765        0.2722        0.2112        0.2418

Best alpha: 0.001  |  OOF RMSE: $21,552

Submission saved: ../../outputs/submissions/sub_v16_pipeline_test.csv



OOF RMSE:      $21,552
Ridge weights: {'xgb': 0.34309961210889856, 'lgbm': 0.3672757101231312, 'et': 0.11978409198547342, 'catboost': 0.17253653535576313}
Best alpha:    0.001
MLflow run logged under experiment "v16-pipeline-test"


In [3]:
print(f'OOF RMSE:      ${results["oof_rmse"]:,.0f}')
print(f'Ridge weights: {results["weights"]}')
print(f'Best alpha:    {results["best_alpha"]}')
print()
print('Per-model mean OOF RMSE:')
import numpy as np
for m, scores in results['fold_scores'].items():
    print(f'  {m:<12} ${np.mean(scores):>10,.0f}')
print()
print('To compare all experiments:')
print('  Run `mlflow ui` in the project root, then open http://localhost:5000')

OOF RMSE:      $21,552
Ridge weights: {'xgb': 0.34309961210889856, 'lgbm': 0.3672757101231312, 'et': 0.11978409198547342, 'catboost': 0.17253653535576313}
Best alpha:    0.001

Per-model mean OOF RMSE:
  xgb          $    21,821
  lgbm         $    21,776
  et           $    23,372
  catboost     $    22,607

To compare all experiments:
  Run `mlflow ui` in the project root, then open http://localhost:5000
